“Transition Compliance Index measures the adequacy of biometric update throughput relative to expected cohort transitions.” (Cannot be done now)

Whether biometric adult updates are keeping pace with the current enrolment/update pressure in the system.

Instead of:

“Expected people turning 18”

We say:

“Expected adult biometric demand implied by current system activity.”

This shifts the metric from demography → operations.

In [1]:
import numpy as np
import pandas as pd

In [2]:
master_table = pd.read_csv("data/aadhaar_master_table.csv", 
                           parse_dates=["date"])

In [3]:
master_table.head()

,state,district,date,age_0_5,age_5_17,age_18_greater,demo_age_5_17,demo_age_17_,bio_age_5_17,bio_age_17_,data_available
0,100000,100000,2025-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,2025-03-01,0.0,0.0,0.0,0.0,0.0,16.0,193.0,True
2,ANDAMAN & NICOBAR ISLANDS,NICOBARS,2025-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
3,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,2025-03-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True
4,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,2025-03-01,0.0,0.0,0.0,32.0,360.0,178.0,101.0,True


### 1. Define expected pressure

In [ ]:
master_table = master_table.sort_values(["state", "district", "date"])

WINDOW = 30  # days

master_table["expected_pressure_30d"] = (
    master_table
    .groupby(["state", "district"])["age_5_17"]
    .rolling(WINDOW, min_periods=10)
    .sum()
    .reset_index(level=[0,1], drop=True)
)

# “How much youth-side activity is the system currently handling?”


In [5]:
master_table["observed_bio_30d"] = (
    master_table
    .groupby(["state", "district"])["bio_age_17_"]
    .rolling(WINDOW, min_periods=10)
    .sum()
    .reset_index(level=[0,1], drop=True)
)


In [6]:
master_table["adult_bio_adequacy_ratio"] = (
    master_table["observed_bio_30d"] /
    master_table["expected_pressure_30d"]
)


In [7]:
district_adequacy = (
    master_table
    .groupby(["state", "district"])["adult_bio_adequacy_ratio"]
    .median()
    .reset_index()
)


In [8]:
district_adequacy

,state,district,adult_bio_adequacy_ratio
0,100000,100000,0.0
1,ANDAMAN & NICOBAR ISLANDS,ANDAMANS,inf
2,ANDAMAN & NICOBAR ISLANDS,NICOBARS,inf
3,ANDAMAN & NICOBAR ISLANDS,SOUTH ANDAMAN,inf
4,ANDAMAN AND NICOBAR ISLANDS,NICOBAR,inf
...,...,...,...
1090,WEST BENGAL,WEST MEDINIPUR,inf
1091,WEST BENGAL,WEST MIDNAPORE,inf
1092,WEST BENGLI,HOOGHLY,NaN
1093,WESTBENGAL,HOOGHLY,6.0


“Due to limited temporal coverage, we do not model individual cohort aging. Instead, we assess whether adult biometric update throughput is adequate relative to contemporaneous enrolment pressure, as an operational proxy for transition readiness.”